In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [148]:
import heapq

class TokenEmbeddings(nn.Module):
    def __init__(self, vocab_size: int, emb_size: int):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, emb_size)

    def forward(self, x: torch.Tensor):
        return self.embeddings(x)

class PositionalEmbeddings(nn.Module):
    def __init__(self, max_seq_len:  int, emb_size: int):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.emb_size = emb_size

        self.embeddings = nn.Embedding(max_seq_len, emb_size)

    def forward(self, seq_len: int):
        return self.embeddings.weight[:seq_len]

import math 

class HeadAttention(nn.Module):
    def __init__(self, emb_size: int, head_size: int, max_seq_len: int):
        super().__init__()
        self.emb_size = emb_size
        self.head_size = head_size
        self.max_seq_len = max_seq_len

        self.W_k = nn.Linear(self.emb_size, self.head_size)
        self.W_q = nn.Linear(self.emb_size, self.head_size)
        self.W_v = nn.Linear(self.emb_size, self.head_size)

        self.mask = torch.tril(torch.ones(self.max_seq_len, self.max_seq_len))

    def forward(self, x: torch.Tensor):
        batch_size, seq_len, emb_size = x.shape
        key = self.W_k(x)
        query = self.W_q(x)
        value = self.W_v(x)

        attention_matrix : torch.Tensor = query @ key.transpose(1,2) / math.sqrt(self.head_size)

        mask = self.mask[:seq_len, :seq_len] == 0
        
        attention_matrix.masked_fill_(mask, float('-inf'))
        attention_matrix = torch.softmax(attention_matrix, 2) 
        
        return attention_matrix @ value
        


class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads: int, emb_size: int,
                head_size: int, max_seq_len: int, dropout: float = 0.1):
        super().__init__()
        self.num_heads = num_heads
        self.emb_size = emb_size
        self.head_size = head_size
        self.max_seq_len = max_seq_len
        self.dropout = dropout

        self.head_attentions = nn.ModuleList([HeadAttention(self.emb_size, self.head_size, self.max_seq_len) for i in range(self.num_heads)])
        self.otpt = nn.Linear(self.head_size * self.num_heads, self.emb_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor):

        tensors = [ head_attention(x) for head_attention in self.head_attentions]
        tensor = torch.cat(tensors, dim=2)
        tensor = self.otpt(tensor)
        tensor = self.dropout(tensor)
        return tensor

class FeedForward(nn.Module):
    def __init__(self, emb_size: int, dropout: float == 0.1):
        super().__init__()
        self.emb_size = emb_size
        self.dropout = dropout

        self.first_linear = nn.Linear(emb_size, emb_size * 4)
        self.relu = nn.ReLU()
        self.second_linear = nn.Linear(4 * emb_size, emb_size)
        self.drpt = nn.Dropout(self.dropout)

    def forward(self, x: torch.Tensor):
        tensor = self.first_linear(x)
        tensor = self.relu(tensor)
        tensor = self.second_linear(tensor)
        tensor = self.drpt(tensor)
        return tensor

class Decoder(nn.Module):
    def __init__(self, num_heads: int, emb_size: int,
                head_size: int, max_seq_len: int, dropout: float = 0.1):
        super().__init__()
        self.num_heads = num_heads
        self.emb_size = emb_size
        self.head_size = head_size
        self.max_seq_len = max_seq_len
        self.dropout = dropout

        self.multi_head_attention = MultiHeadAttention(self.num_heads, self.emb_size, self.head_size, self.max_seq_len, self.dropout)
        self.feed_forward = FeedForward(self.emb_size, self.dropout)
        self.first_norm = nn.LayerNorm(self.emb_size)
        self.second_norm = nn.LayerNorm(self.emb_size)

    def forward(self, x: torch.Tensor):
        tensor = self.multi_head_attention(x)
        tensor += x
        tensor = self.first_norm(tensor)
        tensor += self.feed_forward(tensor)
        tensor = self.second_norm(tensor)
        return tensor


class GPT(nn.Module):
    def __init__(self, vocab_size: int, max_seq_len: int, emb_size: int,
                num_heads: int, head_size: int, num_layers: int,
                dropout: float = 0.1, device: str = "cpu"):
        super().__init__()
        self.vocab_size = vocab_size
        self.max_seq_len = max_seq_len
        self.emb_size = emb_size
        self.num_heads = num_heads
        self.head_size = head_size
        self.num_layers = num_layers
        self.dropout = dropout
        self.device = device

        self.token_embeddings = TokenEmbeddings(self.vocab_size, self.emb_size)
        self.positional_embeddings = PositionalEmbeddings(self.max_seq_len, self.emb_size)
        self.drpt = nn.Dropout(self.dropout)
        self.decoders = nn.Sequential(*[Decoder(self.num_heads, self.emb_size, self.head_size, self.max_seq_len, self.dropout) for i in range(self.num_layers)])
        self.linear = nn.Linear(self.emb_size, self.vocab_size)

    def forward(self, x: torch.Tensor):
        tensor = self.token_embeddings(x) + self.positional_embeddings(x.shape[1])
        tensor = self.drpt(tensor)
        tensor = self.decoders(tensor)
        tensor = self.linear(tensor)
        return tensor
        
    def generate(self, x: torch.Tensor, max_new_tokens: int, do_sample: bool, temperature: float = 1.0,
                top_k: int = None, top_p: float = None):

        for i in range(max_new_tokens):
            x_hat = x[:, -self.max_seq_len : ]
            logits = self.forward(x_hat)
            logits = logits[:, -1, :]
            logits /= temperature
            if not do_sample:
                probas = logits.softmax(1)
                new_tokens = torch.argmax(probas, dim=1)
            else:
                if top_k is not None:
                    new_logits = []
                    for j in range(logits.shape[0]):
                        logits_j = logits[j,:]
                        topk_values, topk_indicies = torch.topk(logits_j, top_k)
                        mask = torch.ones_like(logits_j) == 1
                        mask[topk_indicies] = 0
                        logits_j.masked_fill_(mask, float('-inf'))
                        new_logits.append(logits_j)
                    logits = torch.stack(new_logits,)
                if top_p is not None:
                    new_logits = []
                    for j in range(logits.shape[0]):
                        logits_j = logits[j,:]
                        probabilities = logits_j.softmax(0)
                        
                        max_probabilities = sorted(probabilities.detach().numpy(),reverse=True)

                        cum_proba = max_probabilities[0]
                        min_proba = max_probabilities[0]
                        m = 1
                        while m < len(max_probabilities):
                            cum_proba += max_probabilities[m]
                            if cum_proba < top_p:
                                min_proba = max_probabilities[m]
                            else:
                                break
                            m += 1
                        mask = probabilities  < min_proba

                        logits_j.masked_fill_(mask, float('-inf'))
                        new_logits.append(logits_j)
                    logits = torch.stack(new_logits,)
                        
                probas = logits.softmax(1)
                new_tokens = torch.multinomial(probas, 1)
            new_tokens = torch.reshape(new_tokens, (new_tokens.shape[0], 1))
            x = torch.cat([x, new_tokens], dim=1)
        return x


In [149]:
GPT(5,6,7,8,9,10).generate(torch.LongTensor([[1,2,3,4]]), max_new_tokens=1, do_sample=True, top_p=0.7)

tensor([[1, 2, 3, 4, 1]])

In [142]:
torch.ones_like(torch.Tensor([[1,2,3],[3,21,1]])) == 1.0

tensor([[True, True, True],
        [True, True, True]])